## Amazon SageMaker Processing jobs

With Amazon SageMaker Processing jobs, you can leverage a simplified, managed experience to run data pre- or post-processing and model evaluation workloads on the Amazon SageMaker platform.

More information on <a href='https://aws.amazon.com/blogs/aws/amazon-sagemaker-processing-fully-managed-data-processing-and-model-evaluation/'> the article about Sagemaker Processing </a>

<img src="Processing-1.jpg">

## Data pre-processing and feature engineering

In [41]:
import boto3
import sagemaker
from sagemaker import get_execution_role
from sagemaker.sklearn.processing import SKLearnProcessor

region = boto3.session.Session().region_name

role = get_execution_role()
sklearn_processor = SKLearnProcessor(framework_version='0.20.0',
                                     role=role,
                                     instance_type='ml.m5.xlarge',
                                     instance_count=1)

In [42]:
import pandas as pd

#input_data = 's3://sagemaker-sample-data-{}/processing/census/census-income.csv'.format(region)

input_data = 's3://alexbc.datalake.us-east-2/heart-disease/heart.csv'
    
df = pd.read_csv(input_data, nrows=10)
df.head(n=10)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1
5,57,1,0,140,192,0,1,148,0,0.4,1,0,1,1
6,56,0,1,140,294,0,0,153,0,1.3,1,0,2,1
7,44,1,1,120,263,0,1,173,0,0.0,2,0,3,1
8,52,1,2,172,199,1,1,162,0,0.5,2,0,3,1
9,57,1,2,150,168,0,1,174,0,1.6,2,0,2,1


In [43]:
from sagemaker.processing import ProcessingInput, ProcessingOutput

sklearn_processor.run(code='simple_preprocessing.py',
                      inputs=[ProcessingInput(
                                  source=input_data,
                                  destination='/opt/ml/processing/input')],
                      
                      outputs=[ProcessingOutput(output_name='train_data',
                                  source='/opt/ml/processing/train'),
                                  ProcessingOutput(output_name='test_data',
                                  source='/opt/ml/processing/test')],
                      
                      arguments=['--train-test-split-ratio', '0.2']
                     )

preprocessing_job_description = sklearn_processor.jobs[-1].describe()

output_data = 's3://alexbc.datalake.us-east-2/heart-disease/'

output_config = preprocessing_job_description['ProcessingOutputConfig']
for output in output_config['Outputs']:
    if output['OutputName'] == 'train_data':
        preprocessed_training_data = output['S3Output']['S3Uri']
    if output['OutputName'] == 'test_data':
        preprocessed_test_data = output['S3Output']['S3Uri']


Job Name:  sagemaker-scikit-learn-2020-03-15-18-11-06-731
Inputs:  [{'InputName': 'input-1', 'S3Input': {'S3Uri': 's3://alexbc.datalake.us-east-2/heart-disease/heart.csv', 'LocalPath': '/opt/ml/processing/input', 'S3DataType': 'S3Prefix', 'S3InputMode': 'File', 'S3DataDistributionType': 'FullyReplicated', 'S3CompressionType': 'None'}}, {'InputName': 'code', 'S3Input': {'S3Uri': 's3://sagemaker-us-east-2-771620814942/sagemaker-scikit-learn-2020-03-15-18-11-06-731/input/code/simple_preprocessing.py', 'LocalPath': '/opt/ml/processing/input/code', 'S3DataType': 'S3Prefix', 'S3InputMode': 'File', 'S3DataDistributionType': 'FullyReplicated', 'S3CompressionType': 'None'}}]
Outputs:  [{'OutputName': 'train_data', 'S3Output': {'S3Uri': 's3://sagemaker-us-east-2-771620814942/sagemaker-scikit-learn-2020-03-15-18-11-06-731/output/train_data', 'LocalPath': '/opt/ml/processing/train', 'S3UploadMode': 'EndOfJob'}}, {'OutputName': 'test_data', 'S3Output': {'S3Uri': 's3://sagemaker-us-east-2-771620814

In [45]:
training_features = pd.read_csv(preprocessed_training_data + '/train_features.csv', nrows=10)
print('Training features shape: {}'.format(training_features.shape))
print(preprocessed_training_data)
training_features.head(n=10)

Training features shape: (10, 14)
s3://sagemaker-us-east-2-771620814942/sagemaker-scikit-learn-2020-03-15-18-11-06-731/output/train_data


,58,1,0,150,270,0.1,0.2,111,1.1,0.8,2,0.3,3,0.4
0,50,1,2,129,196,0,1,163,0,0.0,2,0,2,1
1,42,1,2,120,240,1,1,194,0,0.8,0,0,3,1
2,62,0,0,160,164,0,0,145,0,6.2,0,3,3,0
3,62,1,2,130,231,0,1,146,0,1.8,1,3,3,1
4,64,1,3,110,211,0,0,144,1,1.8,1,0,2,1
5,59,1,0,140,177,0,1,162,1,0.0,2,1,3,0
6,35,1,0,120,198,0,1,130,1,1.6,1,0,3,0
7,57,1,2,150,126,1,1,173,0,0.2,2,1,3,1
8,48,1,0,122,222,0,0,186,0,0.0,2,0,2,1
9,52,1,2,138,223,0,1,169,0,0.0,2,4,2,1
